# Momento Evaluativo II: Redes Neuronales Convolucionales

**Institución Universitaria ITM** · Facultad de Ingenierías
**Materia:** Introducción al aprendizaje profundo · **Docente:** Kevin Osorno Castillo
**Fecha límite de entrega:** martes 13 de octubre de 2026

**Integrantes:** Maria Jose Yepez, Franyelica Garcia

---

**Cumplimiento de las restricciones del enunciado**

- Dataset distinto de CIFAR-10 y de los usados en las demostraciones de clase.
- CNN **propia**, construida solo con capas básicas de PyTorch (`Conv2d`, `BatchNorm2d`, `ReLU`, `MaxPool2d`, `AdaptiveAvgPool2d`, `Linear`, `Dropout`).
- **Sin** transfer learning, **sin** modelos preentrenados y **sin** arquitecturas CNN importadas.
- **Sin** data augmentation: las únicas transformaciones son cambio de tamaño, conversión a tensor y normalización.
- El conjunto de **prueba** se usa una sola vez, al final, tras fijar la arquitectura y el proceso de entrenamiento.

**Cómo ejecutarlo:** `Kernel → Restart & Run All`. Requiere `torch`, `torchvision`, `scikit-learn`, `matplotlib`, `pillow` y `numpy`.

In [ ]:
DATASET_NOMBRE = "COMPLETAR (nombre exacto del dataset)"
FUENTE_URL     = "COMPLETAR (enlace desde el que se descargó el dataset)"
ROOT           = r"C:\Users\Usuario\Documents\GitHub\gatos-perros_cnn\archive\images"   # carpeta que contiene los .jpg
# =======================================================================

import os, re, json, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, accuracy_score)

# Subconjunto de clases utilizado (se justifica en la sección 1)
CLASES = ["Abyssinian", "Bengal", "Persian", "Siamese", "Sphynx",
          "beagle", "basset_hound", "pug", "samoyed", "shiba_inu"]

SIZE       = 64      # las imágenes se redimensionan a SIZE x SIZE
BATCH_SIZE = 64
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Integrantes:", ", ".join(INTEGRANTES))
print("Dispositivo:", device, "| PyTorch:", torch.__version__)

## 1. Selección del conjunto de datos

En la celda siguiente se calculan, a partir de los archivos, los datos que pide el enunciado: nombre, fuente, número total de imágenes, número de clases, imágenes por clase y dimensiones originales.

In [ ]:
# Las etiquetas están en el nombre del archivo: "Abyssinian_12.jpg" -> clase "Abyssinian"
patron   = re.compile(r"^(.+)_\d+\.jpg$", re.IGNORECASE)
archivos = sorted(f for f in os.listdir(ROOT) if patron.match(f))
otros    = [f for f in os.listdir(ROOT) if not patron.match(f)]
clase_de = lambda f: patron.match(f).group(1)

conteo_total = Counter(clase_de(f) for f in archivos)

print("1. Nombre del dataset:", DATASET_NOMBRE)
print("2. Fuente y enlace   :", FUENTE_URL)
print("3. Total de imágenes del dataset descargado:", len(archivos))
print("4. Número de clases del dataset descargado :", len(conteo_total))
print(f"   (archivos que no son imágenes .jpg e ignorados: {len(otros)}, p. ej. {otros[:3]})")

print("\n5. Imágenes por clase (dataset completo):")
for c, n in sorted(conteo_total.items()):
    print(f"   {c:28s} {n}")

# ---- Subconjunto realmente usado ----
total_sub = sum(conteo_total[c] for c in CLASES)
print(f"\nSUBCONJUNTO USADO EN ESTE TRABAJO: {len(CLASES)} clases, {total_sub} imágenes")
for c in CLASES:
    print(f"   {c:28s} {conteo_total[c]}")

In [ ]:
# 6. Dimensiones originales (solo se lee la cabecera de cada imagen)
anchos, altos, modos, corruptas = [], [], Counter(), []
for f in archivos:
    try:
        with Image.open(os.path.join(ROOT, f)) as img:
            anchos.append(img.width); altos.append(img.height); modos[img.mode] += 1
    except Exception:
        corruptas.append(f)

anchos, altos = np.array(anchos), np.array(altos)
print("Imágenes que no se pudieron abrir:", len(corruptas))
print("Modos de color:", dict(modos), "(se unifican a RGB en el preprocesamiento)")
print(f"Ancho: mín {anchos.min()}, máx {anchos.max()}, mediana {np.median(anchos):.0f}")
print(f"Alto : mín {altos.min()}, máx {altos.max()}, mediana {np.median(altos):.0f}")
print("Tamaños distintos:", len(set(zip(anchos.tolist(), altos.tolist()))))
print("Más frecuentes:", Counter(zip(anchos.tolist(), altos.tolist())).most_common(3))

In [ ]:
# 7. Ejemplos visuales (una imagen por clase del subconjunto) y distribución por clase
ncols = 5
nrows = int(np.ceil(len(CLASES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows))
for ax in axes.ravel():
    ax.axis("off")
for ax, c in zip(axes.ravel(), CLASES):
    f = next(f for f in archivos if clase_de(f) == c)
    with Image.open(os.path.join(ROOT, f)) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(c, fontsize=10)
plt.suptitle("Ejemplos del dataset (tamaño original)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 3))
plt.bar(CLASES, [conteo_total[c] for c in CLASES])
plt.xticks(rotation=45, ha="right"); plt.ylabel("Imágenes"); plt.title("Imágenes por clase (subconjunto)")
plt.tight_layout(); plt.show()

### ¿Por qué este dataset puede abordarse con una CNN?

- Es un problema de **clasificación de imágenes RGB**, donde la información relevante (bordes, texturas de pelaje, formas de orejas y hocico) es **local** y se repite en distintas posiciones de la imagen. Las convoluciones están diseñadas para explotar justamente esa estructura espacial.
- Los filtros **comparten pesos** en toda la imagen, lo que reduce de forma drástica el número de parámetros frente a una red completamente conectada (se cuantifica en la pregunta 3 de la sección 7).
- Las capas apiladas permiten aprender una **jerarquía de características** (bordes → texturas → partes → objeto) sin diseñarlas a mano.
- El pooling aporta cierta tolerancia a pequeños desplazamientos del animal dentro de la foto.

### Justificación del subconjunto de clases

Se trabaja con **10 de las 37 razas disponibles** (5 de gato y 5 de perro, unas 2.000 imágenes en total) para que el entrenamiento sea viable y reproducible en el equipo disponible, manteniendo clases balanceadas (200 imágenes cada una). La selección incluye razas visualmente **cercanas entre sí** (p. ej. *Abyssinian*/*Bengal*, *beagle*/*basset_hound*) y otras muy distintas (p. ej. *pug*, *samoyed*), lo que permite analizar después qué confunde el modelo. Es una decisión de diseño y una limitación: con solo 200 imágenes por clase y sin data augmentation, el modelo dispone de pocos ejemplos.

## 2. Preparación de los datos

Decisiones de preparación:

- **División** estratificada **70 % entrenamiento / 15 % validación / 15 % prueba**, con semilla fija (`SEED = 42`) para que sea reproducible.
- **Dimensiones consistentes:** todas las imágenes se convierten a RGB y se redimensionan a **64 × 64** (como casi todas son 4:3 o 3:4, la imagen se deforma ligeramente al volverla cuadrada).
- **Conversión a tensor** (valores en [0, 1]) y **normalización** por canal con la media y la desviación estándar calculadas **solo con el conjunto de entrenamiento**, para no filtrar información de validación ni de prueba.
- **No se aplica ninguna transformación aleatoria** (sin data augmentation). El `shuffle` del `DataLoader` de entrenamiento solo cambia el orden de los lotes, no modifica las imágenes.
- Las imágenes se cargan y redimensionan **una sola vez** y quedan en memoria, para que cada época sea rápida.

In [ ]:
def listar_muestras(root, clases):
    muestras = []
    for f in sorted(os.listdir(root)):
        m = patron.match(f)
        if m is None or m.group(1) not in clases:
            continue
        muestras.append((os.path.join(root, f), m.group(1)))
    nombres = sorted(clases)
    c2i = {c: i for i, c in enumerate(nombres)}
    return [(p, c2i[c]) for p, c in muestras], nombres


class PetDataset(Dataset):
    def __init__(self, muestras, size, mean=None, std=None):
        redim = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.PILToTensor(),                      # uint8 [3, H, W]
        ])
        imgs = []
        for ruta, _ in muestras:
            with Image.open(ruta) as im:
                imgs.append(redim(im.convert("RGB")))
        self.images = torch.stack(imgs)                    # [N, 3, H, W] uint8
        self.labels = torch.tensor([y for _, y in muestras])
        self.mean, self.std = mean, std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i].float() / 255.0                 # a tensor en [0, 1]
        if self.mean is not None:
            x = (x - self.mean) / self.std                 # normalización
        return x, self.labels[i]


def crear_loaders(root, size, batch_size, clases):
    muestras, nombres = listar_muestras(root, clases)

    # 70 / 15 / 15 estratificado y reproducible
    train_m, temp_m = train_test_split(
        muestras, test_size=0.30, stratify=[y for _, y in muestras], random_state=SEED)
    val_m, test_m = train_test_split(
        temp_m, test_size=0.50, stratify=[y for _, y in temp_m], random_state=SEED)

    train_ds = PetDataset(train_m, size)
    val_ds   = PetDataset(val_m, size)
    test_ds  = PetDataset(test_m, size)

    # Media y desviación SOLO con entrenamiento
    x = train_ds.images.float() / 255.0
    mean = x.mean(dim=(0, 2, 3)).view(3, 1, 1)
    std  = x.std(dim=(0, 2, 3)).view(3, 1, 1)
    del x
    for ds in (train_ds, val_ds, test_ds):
        ds.mean, ds.std = mean, std

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_dl, val_dl, test_dl, nombres, mean, std


train_dl, val_dl, test_dl, nombres, mean, std = crear_loaders(ROOT, SIZE, BATCH_SIZE, CLASES)

n_tr, n_va, n_te = (len(d.dataset) for d in (train_dl, val_dl, test_dl))
n_total = n_tr + n_va + n_te
print(f"Entrenamiento: {n_tr} ({100*n_tr/n_total:.0f} %) | Validación: {n_va} ({100*n_va/n_total:.0f} %) | Prueba: {n_te} ({100*n_te/n_total:.0f} %)")
print("Media (train):", [round(v, 4) for v in mean.flatten().tolist()])
print("Std   (train):", [round(v, 4) for v in std.flatten().tolist()])

for nombre_split, dl in [("train", train_dl), ("val", val_dl), ("test", test_dl)]:
    cnt = Counter(dl.dataset.labels.tolist())
    print(f"Imágenes por clase en {nombre_split:5s}:", [cnt[i] for i in range(len(nombres))])

x, y = next(iter(train_dl))
print("\nTensor de entrada [B, C, H, W]:", tuple(x.shape), x.dtype)
print("Etiquetas [B]:", tuple(y.shape), y.dtype)

**Dimensiones finales de los tensores de entrada: `[B, C, H, W] = [64, 3, 64, 64]`**

- **B** (batch): número de imágenes que la red procesa a la vez (64; el último lote de cada época puede ser menor).
- **C** (canales): 3, los canales de color R, G y B.
- **H** (alto) y **W** (ancho): 64 y 64 píxeles tras el cambio de tamaño.

## 3. Diseño de la arquitectura CNN

Se diseña una CNN propia con **4 bloques convolucionales** (`Conv2d → BatchNorm2d → ReLU → MaxPool2d`) y una etapa de clasificación completamente conectada. La clase admite dos variantes del clasificador, que se comparan en la sección 4:

- `use_gap=False`: `Flatten` del mapa `[B, 256, 4, 4]` → vector de 4096 → `Linear(4096, 128)`.
- `use_gap=True` (**modelo final**): *global average pooling* (`AdaptiveAvgPool2d(1)`) → vector de 256 → `Linear(256, 128)`.

**Decisiones de diseño**

- Kernel 3×3 con `padding=1` y `stride=1`: la convolución conserva el tamaño espacial y es el pooling el que lo reduce a la mitad en cada bloque.
- El número de filtros se duplica (32 → 64 → 128 → 256) mientras el mapa se reduce, para compensar la pérdida de resolución espacial con mayor riqueza de características.
- **BatchNorm** estabiliza el entrenamiento; **Dropout(0.5)** en el clasificador reduce el sobreajuste, que es el principal riesgo con pocas imágenes y sin data augmentation.

In [ ]:
class PetCNN(nn.Module):
    def __init__(self, num_classes=10, use_gap=False, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),     # [B, 32, 32, 32]

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),     # [B, 64, 16, 16]

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),    # [B, 128, 8, 8]

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2),    # [B, 256, 4, 4]
        )

        if use_gap:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),                           # [B, 256, 1, 1]
                nn.Flatten(),                                      # [B, 256]
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes),                       # [B, num_classes]
            )
        else:
            self.classifier = nn.Sequential(
                nn.Flatten(),                                      # [B, 4096]
                nn.Linear(256 * 4 * 4, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes),
            )

    def forward(self, x):
        return self.classifier(self.features(x))


modelo_final = PetCNN(num_classes=len(CLASES), use_gap=True)
print(modelo_final)

print("\nDimensión de salida tras cada capa (entrada de ejemplo [1, 3, 64, 64]):")
modelo_final.eval()
t = torch.randn(1, 3, SIZE, SIZE)
for capa in list(modelo_final.features) + list(modelo_final.classifier):
    t = capa(t)
    print(f"  {capa.__class__.__name__:18s} -> {tuple(t.shape)}")

p_total = sum(p.numel() for p in modelo_final.parameters() if p.requires_grad)
p_conv  = sum(p.numel() for p in modelo_final.features.parameters())
p_clf   = sum(p.numel() for p in modelo_final.classifier.parameters())
print(f"\nParámetros entrenables: {p_total:,} (bloques convolucionales: {p_conv:,} | clasificador: {p_clf:,})")

**Tabla de la arquitectura final** (`use_gap=True`; `B` = tamaño del batch, `Conv` = kernel 3×3, stride 1, padding 1)

| Capa | Configuración | Dimensión de salida |
|---|---|---|
| Entrada | Imagen RGB 64×64 normalizada | `[B, 3, 64, 64]` |
| Conv1 + BN + ReLU | 32 filtros (3→32 canales) | `[B, 32, 64, 64]` |
| Pool1 | MaxPool 2×2, stride 2 | `[B, 32, 32, 32]` |
| Conv2 + BN + ReLU | 64 filtros (32→64 canales) | `[B, 64, 32, 32]` |
| Pool2 | MaxPool 2×2, stride 2 | `[B, 64, 16, 16]` |
| Conv3 + BN + ReLU | 128 filtros (64→128 canales) | `[B, 128, 16, 16]` |
| Pool3 | MaxPool 2×2, stride 2 | `[B, 128, 8, 8]` |
| Conv4 + BN + ReLU | 256 filtros (128→256 canales) | `[B, 256, 8, 8]` |
| Pool4 | MaxPool 2×2, stride 2 | `[B, 256, 4, 4]` |
| Global Avg Pool | `AdaptiveAvgPool2d(1)` | `[B, 256, 1, 1]` |
| Flatten | | `[B, 256]` |
| Linear1 + ReLU + Dropout | 256 → 128 neuronas, p = 0.5 | `[B, 128]` |
| Salida (Linear2) | 128 → 10 neuronas | `[B, 10]` |

La variante `use_gap=False` sustituye las filas *Global Avg Pool* y *Flatten* por un `Flatten` a `[B, 4096]` y usa `Linear1: 4096 → 128`.

## 4. Entrenamiento

**Hiperparámetros comunes**

- **Función de pérdida:** `CrossEntropyLoss` (clasificación multiclase; la red devuelve logits, sin softmax).
- **Learning rate inicial:** 0.001. **Batch size:** 64.
- La validación se evalúa en cada época con `model.eval()` y sin gradientes (así Dropout y BatchNorm se comportan correctamente).
- Se guarda el modelo de la época con **menor pérdida de validación**. El conjunto de prueba **no se usa** en esta sección.

**Tres experimentos** (cada uno se ejecuta una vez, con la misma semilla)

| Versión | Optimizador | Otros cambios | Clasificador | Épocas |
|---|---|---|---|---|
| **v1** (línea base) | Adam | lr fijo | `Flatten` (4096) | 30 |
| **v2a** | AdamW (weight decay 0.01) | ReduceLROnPlateau (factor 0.5, paciencia 3) + early stopping (paciencia 8) | `Flatten` (4096) | máx. 40 |
| **v2b** | AdamW (weight decay 0.01) | igual que v2a | Global average pooling (256) | máx. 40 |

v2a cambia a la vez el optimizador y el scheduler, por lo que su efecto individual no puede separarse. La comparación v2b frente a v2a aísla el efecto del global average pooling.

In [ ]:
def correr_epoca(model, loader, criterion, optimizer=None):
    """Una pasada por el loader: entrena si hay optimizer; si no, solo evalúa."""
    entrenando = optimizer is not None
    model.train() if entrenando else model.eval()

    loss_total, correctas, total = 0.0, 0, 0
    with torch.set_grad_enabled(entrenando):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if entrenando:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            loss_total += loss.item() * labels.size(0)
            correctas  += (outputs.argmax(dim=1) == labels).sum().item()
            total      += labels.size(0)

    return loss_total / total, 100 * correctas / total


def entrenar(nombre, cfg):
    torch.manual_seed(SEED)
    model = PetCNN(num_classes=len(CLASES), use_gap=cfg["use_gap"]).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n===== {nombre} | parámetros: {n_params:,} | {cfg} =====")

    criterion = nn.CrossEntropyLoss()
    if cfg["optim"] == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                      weight_decay=cfg["weight_decay"])
    scheduler = None
    if cfg["scheduler"]:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)

    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    mejor_loss, mejor_epoca, sin_mejora = float("inf"), 0, 0

    for epoca in range(1, cfg["epocas"] + 1):
        t0 = time.time()
        tr_loss, tr_acc = correr_epoca(model, train_dl, criterion, optimizer)
        va_loss, va_acc = correr_epoca(model, val_dl, criterion)
        lr_actual = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step(va_loss)

        for k, v in zip(hist, (tr_loss, va_loss, tr_acc, va_acc, lr_actual)):
            hist[k].append(v)

        marca = ""
        if va_loss < mejor_loss:
            mejor_loss, mejor_epoca, sin_mejora = va_loss, epoca, 0
            torch.save(model.state_dict(), f"mejor_modelo_{nombre}.pt")
            marca = "  <- mejor"
        else:
            sin_mejora += 1

        print(f"[{nombre}] Epoch {epoca:02d} "
              f"Train Loss: {tr_loss:.4f} Train Acc: {tr_acc:.2f}% | "
              f"Val Loss: {va_loss:.4f} Val Acc: {va_acc:.2f}% | "
              f"lr {lr_actual:.5f} ({time.time() - t0:.1f}s){marca}")

        if cfg["paciencia"] is not None and sin_mejora >= cfg["paciencia"]:
            print(f"Early stopping en la época {epoca}")
            break

    print(f"Mejor época de {nombre} según Val Loss: {mejor_epoca} (Val Loss {mejor_loss:.4f})")
    return {"hist": hist, "mejor_epoca": mejor_epoca, "mejor_val_loss": mejor_loss,
            "params": n_params, "cfg": cfg}


EXPERIMENTOS = {
    "v1":  dict(use_gap=False, optim="adam",  lr=1e-3, weight_decay=0.0,  scheduler=False, paciencia=None, epocas=30),
    "v2a": dict(use_gap=False, optim="adamw", lr=1e-3, weight_decay=1e-2, scheduler=True,  paciencia=8,    epocas=40),
    "v2b": dict(use_gap=True,  optim="adamw", lr=1e-3, weight_decay=1e-2, scheduler=True,  paciencia=8,    epocas=40),
}

resultados = {nombre: entrenar(nombre, cfg) for nombre, cfg in EXPERIMENTOS.items()}

## 5. Análisis del entrenamiento

Se grafican las curvas *Training vs. Validation* de la **versión final (v2b)** y, como referencia, de la línea base (v1).

In [ ]:
def graficar_curvas(nombre):
    r = resultados[nombre]
    h, mejor = r["hist"], r["mejor_epoca"]
    epocas = range(1, len(h["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epocas, h["train_loss"], label="Train Loss")
    plt.plot(epocas, h["val_loss"], label="Val Loss")
    plt.axvline(mejor, color="gray", linestyle="--", label=f"Mejor época ({mejor})")
    plt.xlabel("Época"); plt.ylabel("Loss"); plt.title(f"Training vs. Validation Loss ({nombre})"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epocas, h["train_acc"], label="Train Accuracy")
    plt.plot(epocas, h["val_acc"], label="Val Accuracy")
    plt.axvline(mejor, color="gray", linestyle="--", label=f"Mejor época ({mejor})")
    plt.xlabel("Época"); plt.ylabel("Accuracy (%)"); plt.title(f"Training vs. Validation Accuracy ({nombre})"); plt.legend()

    plt.tight_layout()
    plt.savefig(f"curvas_{nombre}.png", dpi=150)
    plt.show()


graficar_curvas("v2b")     # modelo final
graficar_curvas("v1")      # línea base, como referencia

In [ ]:
# Tabla comparativa (valores en la mejor época de cada versión)
print(f"{'Versión':8s}{'Parámetros':>12s}{'Mejor época':>13s}{'Val Loss':>10s}{'Val Acc':>10s}{'Train Acc':>11s}")
for n, r in resultados.items():
    i = r["mejor_epoca"] - 1
    h = r["hist"]
    print(f"{n:8s}{r['params']:>12,}{r['mejor_epoca']:>13d}{h['val_loss'][i]:>10.4f}"
          f"{h['val_acc'][i]:>9.2f}%{h['train_acc'][i]:>10.2f}%")

plt.figure(figsize=(13, 4))
for pos, (clave, titulo) in enumerate([("val_loss", "Validation Loss"), ("val_acc", "Validation Accuracy (%)")], 1):
    plt.subplot(1, 2, pos)
    for n, r in resultados.items():
        plt.plot(range(1, len(r["hist"][clave]) + 1), r["hist"][clave], label=n)
    plt.xlabel("Época"); plt.title(titulo + ": comparación de versiones"); plt.legend()
plt.tight_layout()
plt.savefig("comparacion_versiones.png", dpi=150)
plt.show()

In [ ]:
# Datos que sustentan las respuestas de esta sección (modelo final v2b)
r = resultados["v2b"]; h = r["hist"]; i = r["mejor_epoca"] - 1
ep_acc_max = int(np.argmax(h["val_acc"])) + 1
brecha = h["train_acc"][i] - h["val_acc"][i]
picos = [e + 1 for e in range(1, len(h["val_loss"])) if h["val_loss"][e] > 1.25 * min(h["val_loss"][:e])]

print(f"Mejor época (menor Val Loss): {r['mejor_epoca']} -> Val Loss {h['val_loss'][i]:.4f}, "
      f"Val Acc {h['val_acc'][i]:.2f}%, Train Acc {h['train_acc'][i]:.2f}%")
print(f"Mayor Val Acc de todo el entrenamiento: {max(h['val_acc']):.2f}% en la época {ep_acc_max}")
print(f"Brecha Train Acc - Val Acc en la mejor época: {brecha:.1f} puntos")
print(f"Brecha al final del entrenamiento: {h['train_acc'][-1] - h['val_acc'][-1]:.1f} puntos")
print(f"Épocas en que la Val Loss subió >25 % sobre su mínimo previo (inestabilidad): {picos}")
print(f"Learning rate inicial -> final: {h['lr'][0]:.5f} -> {h['lr'][-1]:.5f}")

### Respuestas de la sección 5

*(Completa y ajusta estas respuestas con tus propias palabras y con los valores impresos en las celdas anteriores.)*

**1. ¿En qué época el modelo presenta su mejor comportamiento sobre datos no utilizados durante el entrenamiento?**
La época se elige por la **menor pérdida de validación**, que es un criterio más estable que la accuracy cuando la validación tiene solo 300 imágenes (cada imagen pesa ≈0,33 puntos de accuracy). Esa es la época guardada en `mejor_modelo_v2b.pt`; su valor exacto y el de la accuracy de validación se imprimen arriba. Si la mayor accuracy de validación ocurre en otra época, se debe a la variabilidad propia de un conjunto de validación pequeño.

**2. ¿Qué diferencias se observan entre entrenamiento y validación?**
La pérdida y la accuracy de **entrenamiento** mejoran de forma casi continua, mientras que las de **validación** mejoran hasta cierto punto y luego se estancan: la brecha entre ambas (impresa arriba) indica **sobreajuste**. La red tiene capacidad para memorizar las imágenes de entrenamiento y, al no usar data augmentation, no tiene cómo evitarlo. Además, la validación es más irregular mientras el learning rate es alto (picos de pérdida); una explicación plausible, no medida aquí, es la combinación de un conjunto de validación pequeño, un learning rate alto y las estadísticas de BatchNorm. Al reducirse el learning rate, las curvas se estabilizan.

**3. ¿Qué modificaciones realizaría para intentar mejorar el modelo?**
Se probaron tres versiones (tabla comparativa de arriba): la línea base (v1), el cambio de optimizador y planificación del learning rate (v2a) y el reemplazo de `Flatten` por *global average pooling* (v2b), que reduce la primera capa densa de ≈524 mil a ≈33 mil parámetros. Cada versión se ejecutó una sola vez, por lo que la variabilidad entre ejecuciones no está medida. Modificaciones pendientes: mayor resolución de entrada (los detalles que distinguen razas parecidas se pierden a 64×64), más regularización, más imágenes por clase o más clases, y repetir los experimentos con varias semillas.

## 6. Evaluación sobre el conjunto de prueba

La arquitectura y el proceso de entrenamiento ya están fijados (versión **v2b**, checkpoint de menor pérdida de validación). **Esta es la única sección que usa el conjunto de prueba**, y se ejecuta una sola vez.

In [ ]:
modelo = PetCNN(num_classes=len(CLASES), use_gap=True).to(device)
modelo.load_state_dict(torch.load("mejor_modelo_v2b.pt", map_location=device))
modelo.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_dl:                     # test_dl no baraja
        salida = modelo(images.to(device))
        y_pred.extend(salida.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(f"Test Accuracy: {100 * accuracy_score(y_true, y_pred):.2f}%  ({(y_true == y_pred).sum()}/{len(y_true)})\n")
print(classification_report(y_true, y_pred, target_names=nombres, digits=3))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=nombres).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Matriz de confusión (conjunto de prueba)")
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=150)
plt.show()

# Pares de clases más confundidos
fuera = [(cm[i, j], nombres[i], nombres[j])
         for i in range(len(nombres)) for j in range(len(nombres)) if i != j]
print("Confusiones más frecuentes (clase real -> clase predicha):")
for n, real, pred in sorted(fuera, reverse=True)[:6]:
    print(f"  {n:2d}  {real} -> {pred}")

In [ ]:
# Cinco imágenes clasificadas correctamente y cinco incorrectamente
# (selección aleatoria con semilla fija, para no escoger solo casos favorables)
rng = np.random.default_rng(0)
aciertos = rng.choice(np.where(y_true == y_pred)[0], 5, replace=False)
errores  = rng.choice(np.where(y_true != y_pred)[0], 5, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(14, 6.5))
for fila, (idxs, titulo) in enumerate([(aciertos, "Correcta"), (errores, "Incorrecta")]):
    for col, i in enumerate(idxs):
        img = test_dl.dataset.images[i].permute(1, 2, 0).numpy()      # imagen 64x64 sin normalizar
        axes[fila, col].imshow(img)
        axes[fila, col].set_title(f"{titulo}\nReal: {nombres[y_true[i]]}\nPred: {nombres[y_pred[i]]}", fontsize=9)
        axes[fila, col].axis("off")
plt.tight_layout()
plt.savefig("aciertos_errores.png", dpi=150)
plt.show()

### Análisis de predicciones y confusiones

**¿Existen clases que el modelo tienda a confundir entre sí?**
*(Responde con la matriz de confusión y la lista de "Confusiones más frecuentes" de arriba: indica qué pares de razas se confunden más, cuántas veces, y por qué crees que ocurre —por ejemplo, razas con pelaje, tamaño u orejas parecidas, o pérdida de detalle a 64×64. Si alguna confusión no tiene una explicación evidente, dilo como observación en lugar de afirmar una causa.)*

**Observaciones sobre las 5 imágenes correctas y las 5 incorrectas**
*(Describe qué tienen en común los errores: pose, fondo, encuadre, color, raza parecida, etc. Esta parte debe basarse en lo que se ve en la figura.)*

## 7. Preguntas de análisis

### Pregunta 1: ¿qué características aprende cada grupo de capas?

El **campo receptivo** (la región de la imagen de entrada que influye en cada posición del mapa) crece al apilar convoluciones y pooling:

In [ ]:
# Campo receptivo teórico de cada capa de la arquitectura (kernel k, stride s)
capas = [("Conv1", 3, 1), ("Pool1", 2, 2), ("Conv2", 3, 1), ("Pool2", 2, 2),
         ("Conv3", 3, 1), ("Pool3", 2, 2), ("Conv4", 3, 1), ("Pool4", 2, 2)]
rf, salto = 1, 1
for nombre_capa, k, s in capas:
    rf    = rf + (k - 1) * salto
    salto = salto * s
    print(f"{nombre_capa}: campo receptivo = {rf} x {rf} píxeles de la imagen 64x64")

In [ ]:
# Filtros aprendidos por la primera capa convolucional (modelo final)
w = modelo.features[0].weight.detach().cpu()               # [32, 3, 3, 3]
w = (w - w.min()) / (w.max() - w.min())                    # a [0, 1] para visualizarlos
fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
for ax, filtro in zip(axes.ravel(), w):
    ax.imshow(filtro.permute(1, 2, 0).numpy())
    ax.axis("off")
plt.suptitle("32 filtros 3x3 de Conv1")
plt.tight_layout()
plt.show()

- **Primeras capas convolucionales (Conv1, Conv2):** su campo receptivo es pequeño, así que responden a patrones locales simples: **bordes, orientaciones, gradientes y contrastes de color**. Los filtros de Conv1 de la figura son directamente interpretables como detectores de este tipo.
- **Capas intermedias (Conv2–Conv3):** combinan esos bordes en **texturas y patrones** (tipo de pelaje, manchas, rayas) y en partes simples de forma (curvas, esquinas).
- **Capas más profundas (Conv4):** con un campo receptivo que abarca más de la mitad de la imagen, pueden responder a **combinaciones de partes** como la forma de la cabeza, el hocico o las orejas, que son rasgos útiles para distinguir razas.

*Nota:* esta descripción es lo que se **espera** de la jerarquía de una CNN; con solo cuatro capas y 64×64, la jerarquía es modesta y solo se verificó visualmente la primera capa.

### Pregunta 2: ¿cuál es el propósito del pooling en esta arquitectura?

La red usa cuatro capas `MaxPool2d(2×2, stride 2)`, que reducen el mapa de 64×64 a 4×4. Sirven para:

- **Reducir el costo** de cómputo y memoria de las capas siguientes (cada pooling divide entre 4 el número de posiciones).
- **Ampliar el campo receptivo**, de modo que las capas profundas "vean" una porción mayor de la imagen.
- **Aportar tolerancia a pequeños desplazamientos** de las características dentro de la imagen.
- **Limitar el número de parámetros** del clasificador.

El costo es la **pérdida de detalle fino**, que probablemente contribuye a las confusiones entre razas parecidas.

### Pregunta 3: ¿qué pasaría con el número de parámetros si se reemplazaran las convoluciones por capas completamente conectadas sobre todos los píxeles?

In [ ]:
entrada      = 3 * SIZE * SIZE                                  # 12.288 valores por imagen
params_conv  = sum(p.numel() for p in modelo_final.features.parameters())
params_cnn   = sum(p.numel() for p in modelo_final.parameters())

# (a) Una sola capa densa que reprodujera la salida de Conv1 (32 mapas de 64x64)
salida_conv1       = 32 * SIZE * SIZE
params_denso_conv1 = entrada * salida_conv1 + salida_conv1
params_conv1       = 3 * 32 * 3 * 3 + 32

# (b) Un MLP que procesa todos los píxeles directamente (entrada -> 512 -> 512 -> clases)
dims = [entrada, 512, 512, len(CLASES)]
params_mlp = sum(a * b + b for a, b in zip(dims[:-1], dims[1:]))

print(f"Entrada aplanada: {entrada:,} valores")
print(f"CNN propia (v2b): {params_cnn:,} parámetros (bloques convolucionales: {params_conv:,})")
print(f"Conv1: {params_conv1:,} parámetros  vs  capa densa equivalente: {params_denso_conv1:,}"
      f"  (≈ {params_denso_conv1 / params_conv1:,.0f} veces más)")
print(f"MLP {dims}: {params_mlp:,} parámetros  (≈ {params_mlp / params_cnn:.1f} veces los de la CNN)")

**Respuesta:** el número de parámetros se dispararía. Cada neurona de una capa completamente conectada necesita un peso por **cada píxel de entrada**, y no existe el reparto de pesos: una convolución reutiliza los mismos pocos pesos (por ejemplo, 3×3×3 por filtro) en todas las posiciones, mientras que una capa densa tiene un peso distinto para cada par píxel-neurona. Las cifras de la celda anterior lo cuantifican: incluso un MLP pequeño sobre los píxeles tiene varias veces más parámetros que toda la CNN, y reproducir solo la primera capa convolucional con una capa densa necesitaría del orden de miles de millones de pesos.

Además de la cantidad, hay otras consecuencias: la red densa ignoraría la **estructura espacial** (un píxel y su vecino se tratan como entradas sin relación) y tendría que **reaprender el mismo patrón en cada posición**. Con solo 1.400 imágenes de entrenamiento, tantos parámetros agravarían el **sobreajuste**.

---

## Conclusiones

*(Escribe aquí 3–5 líneas: resultado final en prueba, qué cambio mejoró más el modelo, principal limitación —sobreajuste, pocas imágenes, resolución— y qué harías después.)*